# Helper Functions


In [1]:
import control as ctrl
import numpy as np

from lib.plots import plot_or_show, plt


def simplify_tf(G, zp_max):

    # Cancela polos/zeros próximos
    G_simplified = ctrl.minreal(G, verbose=False, tol=2.0)

    zeros = ctrl.zeros(G_simplified)
    poles = ctrl.poles(G_simplified)

    # Remove dinâmicas muito rápidas
    zeros_clamped = [z for z in zeros if np.abs(np.real(z)) < zp_max]
    poles_clamped = [p for p in poles if np.abs(np.real(p)) < zp_max]

    # Sistema temporário com ganho unitário
    G_simplified = ctrl.zpk(zeros_clamped, poles_clamped, 1.0)

    # Preserva ganho DC
    k = ctrl.dcgain(G) / ctrl.dcgain(G_simplified)

    return ctrl.zpk(zeros_clamped, poles_clamped, k)


def compare_step_responses(G_original, G_simplified, name, ylabel=None):
    fig, ax = plt.subplots()

    t, y = ctrl.step_response(G_original)
    ax.plot(t, y, label="Original")  # type: ignore

    t, y = ctrl.step_response(G_simplified)
    ax.plot(t, y, "--", label="Simplificada")  # type: ignore

    ax.set_xlabel("Tempo / h")

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    ax.grid(True)
    ax.legend()

    plot_or_show(f"G_vs_G_simplified/{name}")


# Simplify Transfer Functions


In [2]:
import hickle as hkl
import sympy as sp

from lib.plots import pzplot
from lib.utils import G_sp_to_ctrl, input_names, output_names, output_units

G_matrix_sp = hkl.load("../outputs/G.hkl")
n_inputs = G_matrix_sp.shape[1]
n_outputs = G_matrix_sp.shape[0]

G_matrix_ctrl = np.empty(G_matrix_sp.shape, dtype=object)
G_matrix_simplified = np.empty(G_matrix_sp.shape, dtype=object)

for i in range(n_outputs):
    for j in range(n_inputs):
        G_name = f"{output_names[i]}_{input_names[j]}"
        G_ylabel = f"{output_names[i]} / {output_units[i]}"

        G_ctrl = G_sp_to_ctrl(G_matrix_sp[i, j])

        tol = 50.0

        if j == 2 and i >= 3:
            tol = 80.0

        G_simplified = simplify_tf(G_ctrl, zp_max=tol)

        G_matrix_ctrl[i, j] = G_ctrl
        G_matrix_simplified[i, j] = G_simplified

        pzplot(G_ctrl, save_path="pzplots/" + G_name)
        pzplot(G_simplified, save_path="pzplots/" + G_name + "_simplified")

        compare_step_responses(G_ctrl, G_simplified, G_name, G_ylabel)


Plot saved to ../figures/pzplots/T1_Ff1.png
Plot saved to ../figures/pzplots/T1_Ff1_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_Ff1.png
Plot saved to ../figures/pzplots/T1_Ff2.png
Plot saved to ../figures/pzplots/T1_Ff2_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_Ff2.png
Plot saved to ../figures/pzplots/T1_FR.png
Plot saved to ../figures/pzplots/T1_FR_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_FR.png
Plot saved to ../figures/pzplots/T1_Q1.png
Plot saved to ../figures/pzplots/T1_Q1_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_Q1.png
Plot saved to ../figures/pzplots/T1_Q2.png
Plot saved to ../figures/pzplots/T1_Q2_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_Q2.png
Plot saved to ../figures/pzplots/T1_Q3.png
Plot saved to ../figures/pzplots/T1_Q3_simplified.png
Plot saved to ../figures/G_vs_G_simplified/T1_Q3.png
Plot saved to ../figures/pzplots/T1_T0.png
Plot saved to ../figures/pzplots/T1_T0_simplified.p

In [3]:
import re

s = sp.Symbol("s")


def G_ctrl_to_sp(G):
    num = G.num[0][0]
    den = G.den[0][0]

    num_expr = sum(c * s ** (len(num) - i - 1) for i, c in enumerate(num))
    den_expr = sum(c * s ** (len(den) - i - 1) for i, c in enumerate(den))

    return sp.simplify(num_expr / den_expr)


G_matrix_simplified_sp = np.empty(G_matrix_simplified.shape, dtype=object)
for i in range(n_outputs):
    for j in range(n_inputs):
        G_matrix_simplified_sp[i, j] = G_ctrl_to_sp(G_matrix_simplified[i, j])


with open(
    "../outputs/funções_de_transferência_simplificadas.txt", "w", encoding="utf-8"
) as f:
    for i in range(n_outputs):
        for j in range(n_inputs):
            f.write(f"g[{i},{j}] =\n")
            G_simplified = G_matrix_simplified_sp[i, j]
            f.write(str(G_simplified))
            f.write("\n\n")


with open(
    "../outputs/funções_de_transferência_simplificadas.tex", "w", encoding="utf-8"
) as f:
    for i in range(n_outputs):
        for j in range(n_inputs):
            expr = G_matrix_simplified_sp[i, j].evalf(3)
            f.write(f"g_{{{i + 1},{j + 1}}}(s) &= ")

            latex_str = sp.latex(expr)
            latex_str = re.sub(r"(?<=\d)\.(?=\d)", "{,}", latex_str)

            f.write(latex_str)
            f.write(" \\\\ \n")

hkl.dump(G_matrix_simplified_sp, "../outputs/G_simplified.hkl")


/home/silas/workspace/ENGF93/.venv/lib/python3.14/site-packages/hickle/lookup.py:1491: SerializedWarning: 'Mul' type not understood, data is serialized:
  warnings.warn(
